# Step 08 — projecting the healthy volunteers

The modules were built on SLE samples only. That was not a convenience: WGCNA defines modules by
correlation **across samples**, so pooling healthy volunteers in would let the between-group mean
difference manufacture correlation. Any two proteins elevated in disease would correlate whether or
not they are co-regulated, and the result is a large module that is really the group contrast.

But the interesting question is exactly the one pooling destroys: **do some healthy people carry
disease signal, and do some patients carry little?**

Projection answers it. The healthy volunteers are scored on axes they had no part in defining, so
where they land is a measurement rather than a construction.

In [ ]:
source("../src/paths.R")
suppressMessages(library(WGCNA))
w    <- readRDS(art("wgcna_A.rds"))
X    <- w$X; mods <- w$mods
HV   <- read.csv(coh("R_healthy_log2_combat.csv"), row.names = 1, check.names = FALSE)
hvm  <- read.csv(coh("R_healthy_meta.csv"),        row.names = 1, check.names = FALSE)
HV   <- HV[, colnames(X), drop = FALSE]
c(sle = nrow(X), healthy = nrow(HV), proteins = ncol(X))

In [ ]:
# Project samples onto modules that were defined WITHOUT them.
#
# Two things make this a projection rather than a second fit, and both are easy
# to get wrong in a way that silently destroys the answer:
#
#  1. the loadings come from the SLE matrix alone and are never recomputed;
#  2. the new samples are scaled with SLE's per-protein mean and sd, NOT their
#     own. Re-centring on the new samples would move their mean to zero, which
#     is precisely the difference being measured.
project_module <- function(md0, X, mods, NEW) {
  g  <- colnames(X)[mods == md0]
  mu <- colMeans(X[, g, drop = FALSE])
  sdv<- apply(X[, g, drop = FALSE], 2, sd)
  keep <- sdv > 0
  g <- g[keep]; mu <- mu[keep]; sdv <- sdv[keep]

  Zs <- scale(X[, g, drop = FALSE], center = mu, scale = sdv)
  v  <- svd(Zs, nu = 0, nv = 1)$v[, 1]
  e  <- as.vector(Zs %*% v)
  # WGCNA aligns an eigenprotein so it tracks the module's mean abundance.
  if (cor(e, rowMeans(Zs)) < 0) { v <- -v; e <- -e }

  Zn <- scale(as.matrix(NEW[, g, drop = FALSE]), center = mu, scale = sdv)
  list(sle = e, new = as.vector(Zn %*% v))
}

## The gate

Before any result is read: pushing the **SLE** matrix through the projection path must reproduce
what `moduleEigengenes` returns. If it does not, the code is refitting rather than projecting and
every number after it is meaningless.

Correlation is compared in absolute value because PC1's sign is arbitrary and WGCNA's alignment
rule and ours need not agree on it.

In [ ]:
mlist <- setdiff(unique(mods), "grey")
ME    <- moduleEigengenes(X, mods)$eigengenes

# NOTE: not lapply(mlist, project_module, X = X, ...) -- a named X binds to
# lapply's OWN first formal, silently pushing mlist into the FUN slot.
P <- lapply(mlist, function(k) project_module(k, X, mods, HV))
names(P) <- mlist

gate <- sapply(mlist, function(k) abs(cor(P[[k]]$sle, ME[[paste0("ME", k)]])))
cat(sprintf("round-trip correlation: min %.4f, median %.4f, over %d modules\n",
            min(gate), median(gate), length(gate)))
stopifnot(min(gate) > 0.99)
cat("GATE PASSED -- this is a projection, not a refit\n")

## Where the healthy volunteers land

Scores are expressed in **SLE standard deviations**, so 0 is the SLE mean and −1 is one SD below it.

Read the overlap, not just the gap. The claim worth making is not "healthy and SLE differ" — they
differ on almost everything — but **how many individuals sit on the wrong side of the other group's
median**, which is the continuum made countable.

In [ ]:
ifn <- locate_ifn_module(mods, colnames(X))    # by curated ISG overlap; colours are per-fit
tab <- do.call(rbind, lapply(mlist, function(k) {
  s <- P[[k]]$sle; h <- P[[k]]$new
  sd_s <- sd(s)
  data.frame(module = k, n_proteins = sum(mods == k),
             sle_mean = 0, hv_mean = round((mean(h) - mean(s)) / sd_s, 2),
             hv_above_sle_median = sum(h > median(s)),
             sle_below_hv_median = sum(s < median(h)),
             p = signif(wilcox.test(s, h)$p.value, 2))
}))
tab$q <- signif(p.adjust(tab$p, "BH"), 2)
tab <- tab[order(tab$hv_mean), ]
cat("interferon module (by curated ISG overlap):", ifn, "\n\n")
head(tab, 12); tab[tab$module == ifn, ]

**The caveat that has to travel with this table.** ComBat was fitted across all 356 samples with
`mod = NULL` — unsupervised, told nothing about health status. Healthy volunteers are 17% of batch A
and 44% of batch B, so part of the true batch shift is group composition, and the correction absorbs
some of the healthy-vs-SLE difference along with it.

**Every separation reported here is therefore a lower bound.** That is the conservative direction and
the right one for unsupervised discovery, but it means a small `hv_mean` is not evidence of no
difference.

In [ ]:
suppressMessages({library(ComplexHeatmap); library(circlize)})
options(repr.plot.width = 11, repr.plot.height = 6)
top <- head(tab$module[order(-abs(tab$hv_mean))], 6)
top <- union(top, ifn)
df  <- do.call(rbind, lapply(top, function(k) {
  s <- P[[k]]$sle
  rbind(data.frame(module = k, group = "SLE",     z = (s - mean(s)) / sd(s)),
        data.frame(module = k, group = "healthy", z = (P[[k]]$new - mean(s)) / sd(s)))
}))
df$module <- factor(df$module, levels = top)
par(mar = c(7, 4, 3, 1))
boxplot(z ~ group + module, data = df, las = 2, col = c("#B2182B", "#2166AC"),
        xlab = "", ylab = "eigenprotein, SLE SD units", cex.axis = 0.7,
        main = "healthy volunteers projected onto SLE-defined modules")
abline(h = 0, lty = 2)

In [ ]:
saveRDS(list(projection = P, table = tab, ifn = ifn), art("projection_healthy.rds"))
write.csv(tab, art("projection_healthy.csv"), row.names = FALSE)